# Estudo Comparativo de Classificação Acústica Submarina — Dataset IARA
### Notebook 19: Análise de Ortogonalidade, Efeito Oráculo e Grau de Consenso no Random Forest — Classe LARGE

**Autor:** Evandro Rocha (UFRJ / COPPE / PEE)
**Data:** Outubro de 2026

---

## Resumo Executivo da Investigação com Random Forest (RF)

A classe **`LARGE`** (embarcações de grande porte: navios mercantes, graneleiros, petroleiros e porta-contêineres, $> 150\\text{ m}$ até mais de $300\\text{ m}$) foi avaliada em todas as **2.140 instâncias de teste** nos 10 folds oficiais do protocolo $5\\times2$cv com a arquitetura **Random Forest (100 árvores com balanceamento de classes)** nas 4 representações (**MEL, LOFAR, HHT e DEMON**).

O objetivo é confrontar os resultados do SVM (Notebook 13) e do MLP (Notebook 16) com os do Random Forest para investigar os limites máximos de consenso acústico e de resgate exclusivo por modulação de envelope na maior fonte de ruído naval do oceano.

### Principais Descobertas no Random Forest:
1. **O Maior Teto de Oráculo de Todo o Dataset IARA ($89{,}77\\%$ — 1.921 / 2.140 cargueiros):**
   - O Oráculo do Random Forest em `LARGE` quebra todos os recordes anteriores, atingindo quase **$90\\%$ de recall conjunto** (superando SVM com $88{,}18\\%$ e MLP com $88{,}83\\%$).
   - O erro comum irrecuperável cai para apenas **$10{,}23\\%$ (219 áudios)**, a menor perda intrínseca observada em todo o estudo.
2. **A Mais Alta Taxa de Unanimidade Acústica Jamais Registrada ($44{,}91\\%$ — 961 áudios):**
   - Quase metade de todos os cargueiros (**961 navios**) são classificados corretamente de forma simultânea e independente pelas 4 representações espectrais e temporais.
   - Somando com o alto consenso ($\\ge 3/4$), **$66{,}31\\%$ dos cargueiros (1.419 áudios)** são acertados por no mínimo 3 modelos.
3. **DEMON Quebra Recorde Absoluto de Resgates Exclusivos ($130$ navios — $6{,}07\\%$):**
   - O DEMON no Random Forest salvou sozinho **130 navios de grande porte** que LOFAR, MEL e HHT erraram simultaneamente. Em árvores de decisão, a assinatura subsônica de rotação de eixos ($< 2\\text{ Hz}$) de motores 2 tempos isola os cargueiros sem qualquer ambiguidade de energia contínua.
4. **A Votação por Maioria Simples ($\\ge 2/4$) Supera o LOFAR Isolado:**
   - Um ensemble cego por maioria atinge **$78{,}27\\%$ ($1.675$ navios)**, superando o LOFAR isolado ($75{,}65\\%$) em **$+2{,}62$ pontos percentuais**.

In [ ]:
# 1. Configuração do Ambiente e Imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print

# Estilização publication-ready (sem dependência externa)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.titlesize': 15,
    'figure.dpi': 150
})

print("Ambiente configurado com sucesso! Bibliotecas prontas.")

## 2. Metodologia Experimental e Base de Dados de Teste

Avaliação exaustiva sobre todas as **2.140 instâncias de teste** onde a classe real comprovada era **`LARGE` (Classe 2)** nos 10 folds do protocolo oficial $5\\times2$cv com **Random Forest (100 árvores, balanced)** nas 4 representações:


In [ ]:
# 2. Dados Consolidados das Predições do Random Forest na Classe LARGE
total_lrg = 2140

# Desempenho individual no RF
hits_data = {
    'Representacao': ['MEL (256-D)', 'DEMON (253-D)', 'HHT Puro (256-D)', 'LOFAR (1024-D)', 'ORÁCULO RF (União 4 Modos)'],
    'Acertos': [1427, 1460, 1470, 1619, 1921],
    'Recall_Pct': [66.68, 68.22, 68.69, 75.65, 89.77]
}
df_hits = pd.DataFrame(hits_data)

# Grau de consenso no RF
consensus_data = {
    'Nivel_Consenso': [
        '4 de 4 (Unanimidade / Independente)',
        '3 de 4 (Alto Consenso)',
        '2 de 4 (Consenso Parcial / Dividido)',
        '1 de 4 (Frágil / Resgate Exclusivo)',
        '0 de 4 (Erro Comum / Incorrigível)'
    ],
    'Audios': [961, 458, 256, 246, 219],
    'Pct_Total_Large': [44.91, 21.40, 11.96, 11.50, 10.23],
    'Pct_Dentro_Oraculo': [50.03, 23.84, 13.33, 12.81, np.nan]
}
df_consensus = pd.DataFrame(consensus_data)

# Resgates exclusivos por representação no RF
exclusive_data = {
    'Representacao': ['HHT Exclusivo', 'MEL Exclusivo', 'LOFAR Exclusivo', 'DEMON Exclusivo'],
    'Audios_Salvos': [25, 26, 65, 130],
    'Percentual': [1.17, 1.21, 3.04, 6.07]
}
df_exclusive = pd.DataFrame(exclusive_data)

# Efeito Reverso em Relação ao Baseline LOFAR (1.619 acertos, 521 erros)
reverse_data = {
    'Destino_Troca': ['Para MEL', 'Para HHT', 'Para DEMON'],
    'Mantem_do_LOFAR': [1309, 1382, 1223],
    'Perde_do_LOFAR (Efeito Reverso)': [310, 237, 396],
    'Pct_Perda_Acertos (%)': [19.15, 14.64, 24.46],
    'Resgata_Erros_LOFAR': [118, 88, 237],
    'Pct_Resgate_Erros (%)': [22.65, 16.89, 45.49],
    'Saldo_Liquido_Audios': [-192, -149, -159]
}
df_reverse = pd.DataFrame(reverse_data)

# Padrão de confusão quando erra em LARGE no RF
confusion_pattern_data = {
    'Representacao': ['LOFAR (521 erros)', 'MEL (713 erros)', 'HHT (670 erros)', 'DEMON (680 erros)'],
    'Confundido_SMALL_Pct': [53.17, 56.24, 59.70, 81.76],
    'Confundido_MEDIUM_Pct': [42.80, 41.23, 35.07, 15.15],
    'Confundido_AMBIENTAL_Pct': [4.03, 2.52, 5.22, 3.09]
}
df_confusion_pattern = pd.DataFrame(confusion_pattern_data)

display(df_hits)
display(df_consensus)
display(df_reverse)

## 3. O Efeito Oráculo no Random Forest: Desempenho Isolado vs. Teto Teórico

O **LOFAR lidera o desempenho individual com $75{,}65\\%$ de recall (1.619 acertos)**.

No entanto, o **Oráculo RF atinge o recorde de $89{,}77\\%$ de recall (1.921 acertos em 2.140 cargueiros)**:
- Um ganho de **$+14{,}1$ pontos percentuais** sobre o melhor modelo isolado;
- O erro comum (onde as 4 florestas erram simultaneamente) cai para apenas **$10{,}23\\%$ (219 áudios)**, o menor de todo o dataset.

In [ ]:
# 3. Gráfico: Desempenho Isolado vs Teto Teórico do Oráculo em LARGE (RF)
plt.figure(figsize=(10, 5.5))
colors = ['#2980b9', '#95a5a6', '#9b59b6', '#e67e22', '#27ae60']
bars = plt.bar(df_hits['Representacao'], df_hits['Recall_Pct'], color=colors, width=0.55, edgecolor='black', linewidth=0.8)
bars[-1].set_hatch('//')

plt.axhline(89.77, color='#27ae60', linestyle='--', lw=1.5, alpha=0.7, label='Teto do Oráculo RF (89.77%)')
plt.axhline(75.65, color='#e67e22', linestyle=':', lw=1.5, alpha=0.7, label='Melhor Isolado LOFAR (75.65%)')

plt.title('Recall Random Forest na Classe LARGE: Modelos Isolados vs. Teto Teórico do Oráculo', fontweight='bold', pad=12)
plt.ylabel('Recall em LARGE (%)')
plt.ylim(0, 105)

for bar in bars:
    h = bar.get_height()
    plt.annotate(f'{h:.1f}%', xy=(bar.get_x() + bar.get_width()/2, h),
                 xytext=(0, 4), textcoords="offset points", ha='center', va='bottom',
                 fontsize=10.5, fontweight='bold', color='#111')

plt.legend(loc='upper left', frameon=True)
plt.tight_layout()
plt.show()

## 4. Análise de Grau de Consenso e Risco do Efeito Reverso em LARGE (RF)

### 4.1 A Pirâmide de Consenso no Random Forest
- **Unanimidade (4/4 certas): $961$ áudios ($44{,}91\\%$)** — a mais alta registrada no IARA;
- **Alto Consenso (3/4 certas): $458$ áudios ($21{,}40\\%$)**;
- **Consenso Dividido (2/4 certas): $256$ áudios ($11{,}96\\%$)**;
- **Resgate Exclusivo (1/4 certa): $246$ áudios ($11{,}50\\%$)**;
- **Erro Comum (0/4 certas): $219$ áudios ($10{,}23\\%$)**.

### 4.2 O Efeito Reverso a partir do LOFAR (1.619 Acertos)
- **Chavear para MEL:** Perde $310$ e ganha $118$ (saldo líquido: $-192$ áudios);
- **Chavear para HHT:** Perde $237$ e ganha $88$ (saldo: $-149$ áudios);
- **Chavear para DEMON:** Perde $396$ e ganha $237$ (saldo: $-159$ áudios).

Como o LOFAR acerta $75{,}65\\%$, chaveamento cego sem critério reduz acertos globais.

### 4.3 O Sucesso da Votação por Maioria em LARGE (RF)
- **Unanimidade (4/4):** $44{,}91\\%$ ($961$ áudios);
- **Maioria Estrita ($\\ge 3/4$):** $66{,}31\\%$ ($1.419$ áudios);
- **Maioria Simples / Empate ($\\ge 2/4$):** **$78{,}27\\%$ ($1.675$ áudios)** $\\rightarrow$ **Supera o LOFAR isolado ($75{,}65\\%$) em $+2{,}62$ p.p.!**
- **Oráculo ($\\ge 1/4$):** **$89{,}77\\%$ ($1.921$ áudios)**.

In [ ]:
# 4. Visualização do Grau de Consenso e do Efeito Reverso em LARGE (RF)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Painel 1: Pirâmide de Consenso dos 2.140 Áudios
labels_cons = ['4 Entradas\n(Unânime)', '3 Entradas\n(Alto Consenso)', '2 Entradas\n(Dividido)', '1 Entrada\n(Exclusivo/Frágil)', '0 Entradas\n(Erro Comum)']
counts_cons = [961, 458, 256, 246, 219]
pcts_cons = [44.91, 21.40, 11.96, 11.50, 10.23]
colors_cons = ['#27ae60', '#2980b9', '#f39c12', '#e74c3c', '#7f8c8d']

bars1 = ax1.bar(labels_cons, counts_cons, color=colors_cons, edgecolor='black', linewidth=0.8, width=0.55)
ax1.set_title('Grau de Consenso RF em LARGE (Total: 2.140 Áudios no 5x2cv)', fontweight='bold', pad=12)
ax1.set_ylabel('Quantidade de Áudios de LARGE')
ax1.set_ylim(0, 1100)

for bar, pct in zip(bars1, pcts_cons):
    h = bar.get_height()
    ax1.annotate(f'{h} ({pct:.1f}%)', xy=(bar.get_x() + bar.get_width()/2, h),
                 xytext=(0, 4), textcoords="offset points", ha='center', va='bottom',
                 fontsize=9.5, fontweight='bold')

# Painel 2: Simulação do Efeito Reverso a partir do LOFAR (1.619 Acertos no RF)
destinos = ['Para MEL', 'Para HHT', 'Para DEMON']
perdas = [310, 237, 396]
resgates = [118, 88, 237]
x = np.arange(len(destinos))
w = 0.35

r1 = ax2.bar(x - w/2, perdas, w, label='Perde Acertos do LOFAR (Efeito Reverso)', color='#c0392b', alpha=0.9, edgecolor='black')
r2 = ax2.bar(x + w/2, resgates, w, label='Resgata dos Erros do LOFAR (Ganhos)', color='#27ae60', alpha=0.9, edgecolor='black')

ax2.set_title('Balanço do Efeito Reverso no RF ao Trocar de Entrada a partir do LOFAR', fontweight='bold', pad=12)
ax2.set_ylabel('Quantidade de Áudios de LARGE')
ax2.set_xticks(x)
ax2.set_xticklabels(destinos, fontweight='bold', fontsize=11)
ax2.set_ylim(0, 480)
ax2.legend(loc='upper right', frameon=True)

for rect in r1:
    h = rect.get_height()
    ax2.annotate(f'-{h}', xy=(rect.get_x() + rect.get_width()/2, h),
                 xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=9.5, fontweight='bold', color='#900')

for rect in r2:
    h = rect.get_height()
    ax2.annotate(f'+{h}', xy=(rect.get_x() + rect.get_width()/2, h),
                 xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=9.5, fontweight='bold', color='#006400')

plt.tight_layout()
plt.show()

## 5. Resgates Exclusivos no Random Forest em LARGE: A Suprema Hegemonia do DEMON

Em **$246$ áudios de `LARGE` ($11{,}50\\%$)**, três entradas falharam e apenas um modelo de floresta aleatória acertou o cargueiro:

1. **DEMON Isolado (130 navios salvos exclusivamente — $6{,}07\\%$):**
   - É o **maior resgate exclusivo registrado em todo o estudo comparativo**!
   - O DEMON isola com total pureza as frequências de batimento de pás de hélices maciças acopladas a eixos ultralentos ($< 2\\text{ Hz}$).
2. **LOFAR Puro (65 navios salvos exclusivamente — $3{,}04\\%$):**
   - Raias tônicas discretas de maquinário de grande porte.
3. **MEL Isolado (26 navios salvos exclusivamente — $1{,}21\\%$):**
   - Potência espectral agregada de descarga.
4. **HHT Puro (25 navios salvos exclusivamente — $1{,}17\\%$):**
   - Instabilidades transitórias na cavitação de hélice.

In [ ]:
# 5. Gráfico: Resgates Exclusivos na Classe LARGE (RF)
plt.figure(figsize=(9, 4.5))
colors_excl = ['#9b59b6', '#2980b9', '#e67e22', '#16a085']
bars_excl = plt.barh(df_exclusive['Representacao'], df_exclusive['Audios_Salvos'], color=colors_excl, height=0.5, edgecolor='black', linewidth=0.8)

plt.title('Resgates Exclusivos RF na Classe LARGE (Total: 246 Áudios Únicos)', fontweight='bold', pad=12)
plt.xlabel('Número de Áudios Salvos Exclusivamente')
plt.xlim(0, 160)

for idx, (bar, pct) in enumerate(zip(bars_excl, df_exclusive['Percentual'])):
    w = bar.get_width()
    tag = "  <-- CAMPEÃO DE RESGATES EM LARGE" if idx == 3 else ""
    plt.annotate(f'{w} áudios ({pct:.1f}%){tag}', xy=(w, bar.get_y() + bar.get_height()/2),
                 xytext=(5, 0), textcoords="offset points", ha='left', va='center',
                 fontsize=9.5, fontweight='bold', color='#111' if idx != 3 else '#117864')

plt.tight_layout()
plt.show()

## 6. Padrões de Erro em LARGE: Para Onde Cada Entrada Envia o Cargueiro no RF?

| Entrada | Principal Erro | Porcentagem do Erro | Segundo Maior Erro | Diagnóstico Mecânico |
| :--- | :---: | :---: | :---: | :--- |
| **LOFAR** | **`SMALL`** | **$53{,}17\\%$** | `MEDIUM` ($42{,}80\\%$) | Na ausência de raias tônicas fortes, o particionamento de Fourier cai em espectro contínuo leve. |
| **MEL** | **`SMALL`** | **$56{,}24\\%$** | `MEDIUM` ($41{,}23\\%$) | Baixa relação sinal-ruído em áudios distantes assemelha-se a ruído de lanchas. |
| **HHT** | **`SMALL`** | **$59{,}70\\%$** | `MEDIUM` ($35{,}07\\%$) | Decomposição de IMFs de alta frequência espalha energia instantânea. |
| **DEMON** | **`SMALL`** | **$81{,}76\\%$** | `MEDIUM` ($15{,}15\\%$) | Quando a modulação perde o pico subsônico, o espectro residual demodulado assemelha-se a cavitação fraca de motores de popa. |

In [ ]:
# 6. Gráfico: Destino do Erro em LARGE por Representação (RF)
plt.figure(figsize=(10, 5.5))
reps = df_confusion_pattern['Representacao']
sml_err = df_confusion_pattern['Confundido_SMALL_Pct']
med_err = df_confusion_pattern['Confundido_MEDIUM_Pct']
amb_err = df_confusion_pattern['Confundido_AMBIENTAL_Pct']

w = 0.52
p1 = plt.bar(reps, sml_err, w, label='Confundido com SMALL (Lanchas)', color='#3498db', alpha=0.9, edgecolor='black', linewidth=0.6)
p2 = plt.bar(reps, med_err, w, bottom=sml_err, label='Confundido com MEDIUM (Rebocadores)', color='#f39c12', alpha=0.9, edgecolor='black', linewidth=0.6)
p3 = plt.bar(reps, amb_err, w, bottom=np.array(sml_err)+np.array(med_err), label='Confundido com AMBIENTAL (Ruído)', color='#95a5a6', alpha=0.9, edgecolor='black', linewidth=0.6)

plt.title('Padrão de Erro RF na Classe LARGE: Para onde cada entrada envia o cargueiro? (%)', fontweight='bold', pad=12)
plt.ylabel('Distribuição Percentual dos Erros (%)')
plt.ylim(0, 118)
plt.legend(loc='upper right', frameon=True)

for i in range(len(reps)):
    plt.text(i, sml_err[i]/2, f'{sml_err[i]:.1f}%', ha='center', va='center', fontsize=9, fontweight='bold', color='white')
    plt.text(i, sml_err[i] + med_err[i]/2, f'{med_err[i]:.1f}%', ha='center', va='center', fontsize=9, fontweight='bold', color='white')
    if amb_err[i] > 3:
        plt.text(i, sml_err[i] + med_err[i] + amb_err[i]/2, f'{amb_err[i]:.1f}%', ha='center', va='center', fontsize=8.5, fontweight='bold', color='white')

plt.tight_layout()
plt.show()

## 7. Painel Consolidado de Descoberta Científica (RF)

Abaixo exibimos o painel visual oficial de 4 quadrantes para a classe `LARGE` com o modelo Random Forest:

In [ ]:
# 7. Exibição do Gráfico Oficial Consolidado salvo em disco
from IPython.display import Image, display

img_path = 'reports/analise_complementaridade_rf_large_oraculo.png'
if os.path.exists(img_path):
    display(Image(filename=img_path))
else:
    print(f"Imagem {img_path} não encontrada. Execute 'scratch/generate_rf_predictions_and_analysis.py' para gerá-la.")

## 8. Comparativo Trilateral em LARGE: SVM Nyström vs. MLP PyTorch vs. Random Forest

| Dimensão Analítica | SVM Nyström (Notebook 13) | MLP PyTorch (Notebook 16) | Random Forest (Notebook 19) | Conclusão Científica |
| :--- | :---: | :---: | :---: | :--- |
| **Melhor Modelo Isolado** | **$76{,}31\\%$** (LOFAR) | $74{,}72\\%$ (LOFAR) | $75{,}65\\%$ (LOFAR) | LOFAR é a âncora soberana de cargueiros em todas as 3 famílias. |
| **Teto do Oráculo** | $88{,}18\\%$ ($1.887$) | $88{,}83\\%$ ($1.901$) | **$89{,}77\\%$ ($1.921$)** | **O Random Forest atinge o maior teto de LARGE da pesquisa (quase $90\\%$)!** |
| **Taxa de Unanimidade (4/4)** | $36{,}92\\%$ ($790$) | $42{,}99\\%$ ($920$) | **$44{,}91\\%$ ($961$)** | Quase metade de todos os cargueiros é imune à escolha da transformada no RF. |
| **Erro Comum Incorrigível** | $11{,}82\\%$ ($253$) | $11{,}17\\%$ ($239$) | **$10{,}23\\%$ ($219$)** | Menor erro irrecuperável de todo o dataset IARA. |
| **Campeão de Resgates Exclusivos** | DEMON ($83$ navios) | DEMON ($117$ navios) | **DEMON ($130$ navios!)** | **Consagração absoluta:** DEMON é indispensável para navios pesados nas 3 famílias. |
| **Votação Simples ($\\ge 2/4$)** | $77{,}34\\%$ ($+1{,}0$ p.p.) | $78{,}08\\%$ ($+3{,}36$ p.p.) | **$78{,}27\\%$ ($+2{,}62$ p.p.)** | A votação majoritária supera o LOFAR isolado nas 3 arquiteturas! |

---

## 9. A Grande Síntese Tridimensional: SVM vs. MLP vs. Random Forest

Com a conclusão dos 9 notebooks de análise de complementaridade (Notebooks 10 a 19), consolida-se um quadro teórico completo da classificação acústica submarina:

1. **Random Forest se consagra como o campeão dos Tetos de Oráculo:**
   - Atinge o maior oráculo em `SMALL` (**$86{,}52\\%$**) e em `LARGE` (**$89{,}77\\%$**);
   - Possui a mais alta taxa de unanimidade em `LARGE` ($44{,}91\\%$);
2. **O DEMON é a maior descoberta de complementaridade transversal do projeto:**
   - Em todas as 3 famílias (SVM, MLP, RF), o DEMON lidera os resgates exclusivos em cargueiros de grande porte ($83$ no SVM, $117$ no MLP, $130$ no RF);
   - No Random Forest, o DEMON também lidera os resgates em lanchas rápidas ($178$ lanchas salvas);
3. **A Hierarquia de Robustez Acústica:**
   - `SMALL` ($\\approx 5	ext{--}17\\%$ unanimidade) $\\ll$ `MEDIUM` ($\\approx 27	ext{--}30\\%$ unanimidade) $\\ll$ `LARGE` ($\\approx 37	ext{--}45\\%$ unanimidade);
   - Prova definitiva de que o ruído hidrodinâmico de marulho penaliza assimetricamente embarcações leves, exigindo fusão multimodal adaptativa.